<a href="https://www.kaggle.com/code/aabdollahii/downloading-and-searching-about-kg?scriptVersionId=332481908" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!wget https://dumps.wikimedia.org/fawiki/latest/fawiki-latest-pages-articles-multistream.xml.bz2

--2026-07-03 18:06:33--  https://dumps.wikimedia.org/fawiki/latest/fawiki-latest-pages-articles-multistream.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1500763603 (1.4G) [application/octet-stream]
Saving to: ‘fawiki-latest-pages-articles-multistream.xml.bz2’

fawiki-latest-pages 100%[===================>]   1.40G  4.79MB/s    in 4m 54s  

2026-07-03 18:11:28 (4.86 MB/s) - ‘fawiki-latest-pages-articles-multistream.xml.bz2’ saved [1500763603/1500763603]



In [ ]:
import pandas 


In [8]:
import bz2
import xml.etree.ElementTree as ET
import csv
import re
import mwparserfromhell

def clean_text(text):
    """
    پاک‌سازی متون ویکی‌پدیا برای جلوگیری از خراب شدن ساختار CSV
    """
    if not text:
        return ""
    
    # ۱. حذف لینک‌های ویکی مانند [[شیراز|شهر شیراز]] -> شهر شیراز یا شیراز
    text = re.sub(re.compile(r'\[\[([^\]|]*\|)?([^\]]+)\]\]'), r'\2', text)
    
    # ۲. حذف الگوهای درونی مانند {{یادکرد}} یا تگ‌های HTML مثل <ref>
    text = re.sub(re.compile(r'\{\{[^\}]*\}\}'), '', text)
    text = re.sub(re.compile(r'<[^>]+>'), '', text)
    
    # ۳. حذف کاراکترهای خط جدید و اینتر که ساختار سطرها را در CSV می‌شکنند
    text = text.replace("\n", " ").replace("\r", " ")
    
    # ۴. حذف فاصله‌های اضافه و کاراکترهای خاص خرابکار مثل پایپ یا کوتیشن‌ها
    text = text.replace("|", " ").replace('"', "'")
    text = re.sub(re.compile(r'\s+'), ' ', text)
    
    return text.strip()

def test_extract_infoboxes_fixed(xml_path, csv_path, max_pages=100):
    extracted_count = 0
    
    with bz2.BZ2File(xml_path, "rb") as f, open(csv_path, "w", encoding="utf-8", newline="") as csv_f:
        # استفاده از کوتیشن‌های امن برای پنداس
        writer = csv.writer(csv_f, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["Subject", "Predicate", "Object"])
        
        context = ET.iterparse(f, events=("end",))
        
        for _, elem in context:
            if elem.tag.endswith("page"):
                title = elem.find("{*}title").text
                text_elem = elem.find("{*}revision/{*}text")
                
                if title and any(title.startswith(x) for x in ["الگو:", "بحث:", "ویکی‌پدیا:", "مدیاویکی:", "رده:"]):
                    elem.clear()
                    continue
                
                if text_elem is not None and text_elem.text:
                    wikitext = mwparserfromhell.parse(text_elem.text)
                    templates = wikitext.filter_templates()
                    
                    has_infobox = False
                    for t in templates:
                        if "جعبه" in t.name.strip():
                            for p in t.params:
                                pred = p.name.strip()
                                obj = p.value.strip()
                                
                                # تمیزکاری ستون‌ها قبل از ذخیره سازی
                                clean_subject = clean_text(title)
                                clean_predicate = clean_text(pred)
                                clean_object = clean_text(obj)
                                
                                # فیلتر کردن ردیف‌های نامعتبر یا خیلی طولانی/کوتاه
                                if (clean_predicate and clean_object and 
                                    len(clean_object) < 80 and len(clean_object) > 1 and 
                                    not clean_predicate.isdigit()):
                                    
                                    writer.writerow([clean_subject, clean_predicate, clean_object])
                                    has_infobox = True
                    
                    if has_infobox:
                        extracted_count += 1
                
                elem.clear()
                
                if extracted_count >= max_pages:
                    print(f"تعداد {max_pages} صفحه با ساختار کاملاً تمیز استخراج شد.")
                    break

# اجرای مجدد تست با متد اصلاح شده
test_extract_infoboxes_fixed(
    xml_path="fawiki-latest-pages-articles-multistream.xml.bz2", 
    csv_path="test_triples_clean.csv", 
    max_pages=100
)

تعداد 100 صفحه با ساختار کاملاً تمیز استخراج شد.


In [9]:
import pandas as pd

# اضافه کردن صریح انکودینگ utf-8 موقع خواندن فایل
df_clean = pd.read_csv("test_triples_clean.csv", encoding="utf-8")
print(f"تعداد سه‌تایی‌ها: {len(df_clean)}")

# نمایش نمونه‌ها با ساختار درست
df_clean.head(20)

تعداد سه‌تایی‌ها: 1838


,Subject,Predicate,Object
0,سعدی,نام,سعدی شیرازی
1,سعدی,تصویر,Sadi in a Rose garden.jpg
2,سعدی,زمینه فعالیت,شعر و نثر فارسی
3,سعدی,ملیت,ایرانی
4,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری
5,سعدی,محل تولد,شیراز، ایران
6,سعدی,تاریخ مرگ,بین ۶۹۰ تا ۶۹۵
7,سعدی,محل مرگ,شیراز، ایران
8,سعدی,محل زندگی,شیراز، بغداد
9,سعدی,مدفن,سعدیه، شیراز


In [ ]:
import bz2
import xml.etree.ElementTree as ET
import csv
import re
import mwparserfromhell
import time

def clean_text(text):
    """
    پاک‌سازی نهایی متون برای استخراج سه‌تایی‌های تمیز و بدون نویز
    """
    if not text:
        return ""
    
    # ۱. حذف لینک‌های ویکی و الگوهای درونی
    text = re.sub(re.compile(r'\[\[([^\]|]*\|)?([^\]]+)\]\]'), r'\2', text)
    text = re.sub(re.compile(r'\{\{[^\}]*\}\}'), '', text)
    text = re.sub(re.compile(r'<[^>]+>'), '', text)
    
    # ۲. یکدست‌سازی فاصله‌ها و حذف کاراکترهای خرابکار خط جدید و پایپ
    text = text.replace("\n", " ").replace("\r", " ").replace("|", " ").replace('"', "'")
    text = re.sub(re.compile(r'\s+'), ' ', text)
    
    return text.strip()

def extract_full_wikipedia_knowledge(xml_path, csv_path):
    """
    پردازش کامل دامپ ویکی‌پدیای فارسی و استخراج تمام سه‌تایی‌های جعبه اطلاعات
    """
    start_time = time.time()
    page_count = 0
    infobox_page_count = 0
    triple_count = 0
    
    print("شروع فرآیند استخراج دانش از ویکی‌پدیای فارسی... لطفاً صبور باشید.")
    
    with bz2.BZ2File(xml_path, "rb") as f, open(csv_path, "w", encoding="utf-8", newline="") as csv_f:
        # استفاده از کوتیشن‌های امن پایتون برای پنداس
        writer = csv.writer(csv_f, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["Subject", "Predicate", "Object"])
        
        # پارس جریانی برای مصرف رم نزدیک به صفر
        context = ET.iterparse(f, events=("end",))
        
        for _, elem in context:
            if elem.tag.endswith("page"):
                page_count += 1
                title = elem.find("{*}title").text
                text_elem = elem.find("{*}revision/{*}text")
                
                # فیلتر صفحات سیستمی و رده‌ها
                if title and any(title.startswith(x) for x in ["الگو:", "بحث:", "ویکی‌پدیا:", "مدیاویکی:", "رده:", "درگاه:"]):
                    elem.clear()
                    continue
                
                if text_elem is not None and text_elem.text:
                    wikitext = mwparserfromhell.parse(text_elem.text)
                    templates = wikitext.filter_templates()
                    
                    has_infobox = False
                    for t in templates:
                        if "جعبه" in t.name.strip():
                            for p in t.params:
                                pred = p.name.strip()
                                obj = p.value.strip()
                                
                                clean_subject = clean_text(title)
                                clean_predicate = clean_text(pred)
                                clean_object = clean_text(obj)
                                
                                # فیلتر سه‌تایی‌های نامعتبر، خیلی طولانی یا تکراری عددی
                                if (clean_predicate and clean_object and 
                                    len(clean_object) < 80 and len(clean_object) > 1 and 
                                    not clean_predicate.isdigit() and len(clean_subject) < 50):
                                    
                                    writer.writerow([clean_subject, clean_predicate, clean_object])
                                    triple_count += 1
                                    has_infobox = True
                    
                    if has_infobox:
                        infobox_page_count += 1
                
                # آزادسازی حافظه برای جلوگیری از پر شدن RAM کاگل
                elem.clear()
                
                # گزارش‌دهی هر ۵۰,۰۰۰ صفحه برای مانیتور کردن وضعیت پیشرفت
                if page_count % 50000 == 0:
                    elapsed = time.time() - start_time
                    print(f"بررسی {page_count:,} صفحه | صفحات دارای جعبه اطلاعات: {infobox_page_count:,} | کل سه‌تایی‌ها: {triple_count:,} | زمان گذشته: {elapsed:.1f} ثانیه")
        
    total_time = time.time() - start_time
    print("\n--- عملیات با موفقیت به پایان رسید ---")
    print(f"کل صفحات بررسی شده: {page_count:,}")
    print(f"صفحات استخراج شده: {infobox_page_count:,}")
    print(f"تعداد کل سه‌تایی‌های ذخیره شده در CSV: {triple_count:,}")
    print(f"زمان کل: {total_time/60:.2f} دقیقه")

# اجرای نهایی روی کل دیتای ویکی‌پدیا در کاگل
extract_full_wikipedia_knowledge(
    xml_path="fawiki-latest-pages-articles-multistream.xml.bz2", 
    csv_path="knowledge_graph_triples.csv"
)

شروع فرآیند استخراج دانش از ویکی‌پدیای فارسی... لطفاً صبور باشید.
بررسی 50,000 صفحه | صفحات دارای جعبه اطلاعات: 5,958 | کل سه‌تایی‌ها: 86,197 | زمان گذشته: 243.5 ثانیه
بررسی 150,000 صفحه | صفحات دارای جعبه اطلاعات: 16,939 | کل سه‌تایی‌ها: 200,946 | زمان گذشته: 500.2 ثانیه
بررسی 200,000 صفحه | صفحات دارای جعبه اطلاعات: 32,610 | کل سه‌تایی‌ها: 379,520 | زمان گذشته: 621.4 ثانیه
بررسی 250,000 صفحه | صفحات دارای جعبه اطلاعات: 37,491 | کل سه‌تایی‌ها: 425,826 | زمان گذشته: 761.5 ثانیه
بررسی 300,000 صفحه | صفحات دارای جعبه اطلاعات: 43,767 | کل سه‌تایی‌ها: 498,784 | زمان گذشته: 829.6 ثانیه
بررسی 350,000 صفحه | صفحات دارای جعبه اطلاعات: 43,978 | کل سه‌تایی‌ها: 501,012 | زمان گذشته: 841.1 ثانیه
بررسی 400,000 صفحه | صفحات دارای جعبه اطلاعات: 66,481 | کل سه‌تایی‌ها: 705,024 | زمان گذشته: 939.5 ثانیه
بررسی 450,000 صفحه | صفحات دارای جعبه اطلاعات: 69,673 | کل سه‌تایی‌ها: 746,738 | زمان گذشته: 977.1 ثانیه
بررسی 500,000 صفحه | صفحات دارای جعبه اطلاعات: 72,753 | کل سه‌تایی‌ها: 774,400 | زمان گذشته: 1023